## VISUALIZATION

In [57]:
# General
import numpy as np
from pathlib import Path

# Other
from bokeh.models import Legend
from bokeh.palettes import Category10
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.io import output_notebook
output_notebook()

Loading BokehJS ...

In [58]:
# Directories

outputs = Path(r"C:\Users\Alex\Desktop\Picaso\NN_project\cloudy_spectra_code\outputs")
out_sin = r"C:\Users\Alex\Desktop\Picaso\NN_project\cloudy_spectra_code\outputs\t1450g4f2k1e9.npz"

In [59]:
# Read single file
def _read_npz_perfile(npz_path):
    """
    Return (w_um, F, params dict) from a per-case .npz.
    """
    npz_path = Path(npz_path)
    d = np.load(npz_path, allow_pickle=False)
    W = d["wavelength_um"].astype(float)
    y   = d["y"]
    x   = d["x"]

    F = y.astype(float)
    Teff, logg, fsed, kzz = map(float, x)

    params = dict(Teff=Teff, logg=logg, fsed=fsed, kzz=kzz, filename=npz_path.name)
    return W, F, params

# Quick statistics
def inspect_npz_file(npz_path):
    """
    Print quick stats for a per-case .npz.
    """
    W, F, p = _read_npz_perfile(npz_path)

    print(f"File            : {p['filename']}")
    print(f"Teff [K]        : {p['Teff']:.0f}")
    print(f"logg [cgs]      : {p['logg']:.0f}")
    print(f"f_sed           : {p['fsed']:.2f}")
    print(f"kzz [cm^2 s^-1] : {p['kzz']:.3e}")
    print(f"Wave range      : {W.size} (micron)")

    return dict(wavelength_um=W, F=F, **p)

# Plotting
def plot_npz_file(npz_path, logy=False):
    """
    Plot a single .npz.
    """
    W, F, p = _read_npz_perfile(npz_path)

    title = f"{p['filename']} — Teff={p['Teff']:.0f}, logg={p['logg']:.0f}, f_sed={p['fsed']:.2f}, kzz={p['kzz']:.2e}"
    src = ColumnDataSource(dict(wavelength_um=W, F=F))

    tools = "pan,wheel_zoom,box_zoom,reset,save"
    fig = figure(title=title, x_axis_label="Wavelength (micron)", y_axis_label="F (erg cm^-2 s^-1 cm^-1)",
                 sizing_mode="stretch_width", height=420, tools=tools, output_backend="canvas")
    if logy:
        fig.y_axis_type = "log"

    fig.add_tools(HoverTool(tooltips=[("Wav (micron)", "@wavelength_um{0.000}"), ("F", "@F{0.00e}")], mode="vline"))
    fig.line('wavelength_um', 'F', source=src, line_width=2)
    show(fig)

In [60]:
inspect_npz_file(out_sin)
plot_npz_file(out_sin, logy=False)

File            : t1450g4f2k1e9.npz
Teff [K]        : 1450
logg [cgs]      : 4
f_sed           : 2.00
kzz [cm^2 s^-1] : 1.000e+09
Wave range      : 844 (micron)
